# Добавляем рекламы

Объединяем дневные визиты и регистрации с данными рекламных кампаний. Для дней без рекламы устанавливаем `cost = 0` и `utm_campaign = 'none'`.

In [ ]:
%run ./conversion.ipynb

In [ ]:
ads = load_csv(
    "12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj",
    "data/ads.csv",
    ["date", "utm_source", "utm_medium", "utm_campaign", "cost"],
).copy()

ads["date"] = pd.to_datetime(ads["date"], format="ISO8601", errors="raise")
ads["date_group"] = ads["date"].dt.normalize()
ads["cost"] = pd.to_numeric(ads["cost"], errors="raise")

ads.head()

In [ ]:
daily_conversion = (
    conversion
    .groupby("date_group", as_index=False)
    .agg(
        visits=("visits", "sum"),
        registrations=("registrations", "sum"),
    )
)

ads_grouped = (
    ads
    .groupby(["date_group", "utm_campaign"], as_index=False)
    .agg(cost=("cost", "sum"))
)

In [ ]:
ads_result = daily_conversion.merge(
    ads_grouped,
    on="date_group",
    how="left",
)

ads_result["cost"] = ads_result["cost"].fillna(0)
ads_result["utm_campaign"] = ads_result["utm_campaign"].fillna("none")

ads_result = (
    ads_result[["date_group", "visits", "registrations", "cost", "utm_campaign"]]
    .sort_values("date_group")
    .reset_index(drop=True)
)

ads_result.to_json("./ads.json")
ads_result